In [1]:
# validate the downscaled data using the same framework with all raw datasets

# V1 - 2026-07-29 - create code - jpk

In [2]:
# libraries
import pandas as pd
import numpy as np
import glob
import os
import rasterio
import HydroErr as he
import concurrent.futures
from pyproj import Transformer

#paths
insitu_tmax_raw = pd.read_csv(r"..\Data\Processed\Insitu\insitu_maxtemp_1991_2020.csv")
insitu_tmin_raw = pd.read_csv(r"..\Data\Processed\Insitu\insitu_mintemp_1991_2020.csv")
tmax_train_df = fr"..\Data_Temp\Processed\Split_stations\train_station_tmax.csv"
tmax_test_df = fr"..\Data_Temp\Processed\Split_stations\test_station_tmax.csv"
tmin_train_df = fr"..\Data_Temp\Processed\Split_stations\train_station_tmin.csv"
tmin_test_df = fr"..\Data_Temp\Processed\Split_stations\test_station_tmin.csv"
output_dir = fr"..\Data_Temp\Processed\Daily_bias_correction\Validation_results"

#parameters
datasets = ['ERA5L', 'HAR', 'CHIRTS', 'IS_K', 'DS']
start_year, start_month = 2001, 1
end_year,   end_month   = 2005, 12
start_date = pd.Timestamp(f"{start_year}-{start_month:02d}-01")
end_date = pd.Timestamp(f"{end_year}-{end_month:02d}-01") + pd.offsets.MonthEnd(0)

In [3]:
# 1 Read insitu and separate train-test

# process tmax ==
insitu_tmax_raw["date"]  = pd.to_datetime(insitu_tmax_raw[["year", "month", "day"]])
insitu_tmax_raw  = insitu_tmax_raw.rename(columns={"lat": "y", "long": "x",
    "max_temp": "tmax"})

# Filter for study period
insitu_tmax_daily = insitu_tmax_raw[
    (insitu_tmax_raw["year"]  >= start_year) &
    (insitu_tmax_raw["year"]  <= end_year)
].copy()
insitu_tmax_daily[["x", "y"]] = insitu_tmax_daily[["x", "y"]].round(5)

# Train/test split
train_ids_tmax = set(pd.read_csv(tmax_train_df)["index_no"])
test_ids_tmax = set(pd.read_csv(tmax_test_df)["index_no"])

insitu_tmax_daily_train = insitu_tmax_daily[insitu_tmax_daily["index_no"].isin(train_ids_tmax)].copy()
insitu_tmax_daily_test = insitu_tmax_daily[insitu_tmax_daily["index_no"].isin(test_ids_tmax)].copy()

# Unique test stations — order must stay fixed (iloc used in sampling)
unique_stn_tmax = (
    insitu_tmax_daily_test[['index_no', 'x', 'y']]
    .drop_duplicates('index_no')
    .reset_index(drop=True)
)
stn_points_tmax = list(zip(unique_stn_tmax['x'], unique_stn_tmax['y']))  


# process tmin ==
insitu_tmin_raw["date"]  = pd.to_datetime(insitu_tmin_raw[["year", "month", "day"]])
insitu_tmin_raw  = insitu_tmin_raw.rename(columns={"lat": "y", "long": "x",
    "min_temp": "tmin"})

# Filter for study period
insitu_tmin_daily = insitu_tmin_raw[
    (insitu_tmin_raw["year"]  >= start_year) &
    (insitu_tmin_raw["year"]  <= end_year)
].copy()
insitu_tmin_daily[["x", "y"]] = insitu_tmin_daily[["x", "y"]].round(5)

# Train/test split
train_ids_tmin = set(pd.read_csv(tmin_train_df)["index_no"])
test_ids_tmin = set(pd.read_csv(tmin_test_df)["index_no"])

insitu_tmin_daily_train = insitu_tmin_daily[insitu_tmin_daily["index_no"].isin(train_ids_tmin)].copy()
insitu_tmin_daily_test = insitu_tmin_daily[insitu_tmin_daily["index_no"].isin(test_ids_tmin)].copy()

# Unique test stations — order must stay fixed (iloc used in sampling)
unique_stn_tmin = (
    insitu_tmin_daily_test[['index_no', 'x', 'y']]
    .drop_duplicates('index_no')
    .reset_index(drop=True)
)
stn_points_tmin = list(zip(unique_stn_tmin['x'], unique_stn_tmin['y']))  

In [4]:
#Helper

def extract_date(filepath, dataset):
    """Parse date from filename"""
    fname = os.path.basename(filepath)
    return pd.to_datetime(fname[-12:-4], format='%Y%m%d')


def get_dataset_paths(t_variable):
    return {'ERA5L': fr"..\Data_Temp\Datasets\ERA5L",
    'HAR': fr"..\Data_Temp\Datasets\HAR",
    'CHIRTS':fr"..\Data_Temp\Datasets\CHIRTS",
    'IS_K': fr"..\Data_Temp\Datasets\Kriging_daily\{t_variable}",
    'DS': fr"..\Data_Temp\Processed\Daily_bias_correction\Final_products\{t_variable}"
    }

def build_dataset_results(dataset_paths, datasets, t_variable):
    """Build {dataset}_files dict pre-filtered to analysis period."""
    dataset_results = {}
    for dataset in datasets:
        if dataset in ['ERA5L', 'HAR', 'CHIRTS']:
            files = sorted(glob.glob(os.path.join(dataset_paths[dataset], f"{dataset.lower()}_{t_variable}_*.tif")))
        else:
            files = sorted(glob.glob(os.path.join(dataset_paths[dataset], "*.tif")))
        filtered = []
        for f in files:
            try:
                d = extract_date(f, dataset)
                if start_date <= d <= end_date:
                    filtered.append(f)
            except Exception as e:
                print(f"  Skipping {os.path.basename(f)}: {e}")
        dataset_results[f'{dataset}_files'] = filtered
        print(f"{dataset}: {len(filtered)} files in period")
    return dataset_results

In [5]:
def _sample_file(file, dataset, stn_points, unique_stn, t_variable):
    """Sample all station points from a single raster file."""
    date_obj = extract_date(file, dataset)
    try:
        with rasterio.open(file) as src:
            if src.crs.to_epsg() != 4326:
                transformer = Transformer.from_crs(
                    "EPSG:4326", src.crs, always_xy=True
                )
                pts = [transformer.transform(x, y) for x, y in stn_points]
            else:
                pts = stn_points

            sampled = list(src.sample(pts))

        return [
            {
                'date':              date_obj,
                'stn_id':            unique_stn.iloc[i]['index_no'],
                f'{dataset}_{t_variable}':   v[0],
            }
            for i, v in enumerate(sampled)
        ]
    except Exception as e:
        print(f"  Error processing {os.path.basename(file)}: {e}")
        return []


def extract_daily_datasets(dataset_results, stn_points, unique_stn, datasets, t_variable):
    """
      Sample each file concurrently and merge at daily level
    """
    df_final = None

    for dataset in datasets:
        files = dataset_results.get(f'{dataset}_files', [])
        if not files:
            print(f"Warning: no files found for {dataset}, skipping.")
            continue
        print(f"Processing {dataset} ({len(files)} daily files)...")

        records = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            for result in executor.map(
                lambda f: _sample_file(f, dataset, stn_points, unique_stn, t_variable), files
            ):
                records.extend(result)

        df_dataset = pd.DataFrame(records)
        if df_dataset.empty:
            print(f"  Warning: no data extracted for {dataset}")
            continue

        df_dataset['date'] = pd.to_datetime(df_dataset['date'])
        df_dataset = df_dataset.assign(
            year  = lambda d: d['date'].dt.year,
            month = lambda d: d['date'].dt.month,
            day = lambda d: d['date'].dt.day
        ).drop(columns='date')

        df_final = (
            df_dataset if df_final is None
            else pd.merge(df_final, df_dataset,
                          on=['stn_id', 'year', 'month', 'day'], how='outer')
        )
        print(f"  → merged shape: {df_final.shape}")

    return df_final

In [6]:
# Build dataset results - Tmax
tmax_results = build_dataset_results(get_dataset_paths("tmax"), datasets, "tmax")

df_grid_tmax = extract_daily_datasets(
    tmax_results, stn_points_tmax, unique_stn_tmax, datasets, "tmax" 
)

df_validation_tmax = pd.merge(
    insitu_tmax_daily_test.rename(columns={'index_no': 'stn_id'}),
    df_grid_tmax,
    on=['stn_id', 'year', 'month', 'day'],
    how='inner'
)
df_validation_tmax.to_csv(os.path.join(output_dir, "Tmax_validation_df.csv"), index = False)

# Build dataset results - Tmin
tmin_results = build_dataset_results(get_dataset_paths("tmin"), datasets, "tmin")

df_grid_tmin = extract_daily_datasets(
    tmin_results, stn_points_tmin, unique_stn_tmin, datasets, "tmin" 
)

df_validation_tmin = pd.merge(
    insitu_tmin_daily_test.rename(columns={'index_no': 'stn_id'}),
    df_grid_tmin,
    on=['stn_id', 'year', 'month', 'day'],
    how='inner'
)
df_validation_tmin.to_csv(os.path.join(output_dir, "Tmin_validation_df.csv"), index = False)

ERA5L: 1826 files in period
HAR: 1826 files in period
CHIRTS: 1826 files in period
IS_K: 1826 files in period
DS: 1826 files in period
Processing ERA5L (1826 daily files)...
  → merged shape: (41998, 5)
Processing HAR (1826 daily files)...
  → merged shape: (41998, 6)
Processing CHIRTS (1826 daily files)...
  → merged shape: (41998, 7)
Processing IS_K (1826 daily files)...
  → merged shape: (41998, 8)
Processing DS (1826 daily files)...
  → merged shape: (41998, 9)
ERA5L: 1826 files in period
HAR: 1826 files in period
CHIRTS: 1826 files in period
IS_K: 1826 files in period
DS: 1826 files in period
Processing ERA5L (1826 daily files)...
  → merged shape: (41998, 5)
Processing HAR (1826 daily files)...
  → merged shape: (41998, 6)
Processing CHIRTS (1826 daily files)...
  → merged shape: (41998, 7)
Processing IS_K (1826 daily files)...
  → merged shape: (41998, 8)
Processing DS (1826 daily files)...
  → merged shape: (41998, 9)


In [7]:
#column normalization function
def normalize_column(col):
    return (col - col.min()) / (col.max() - col.min())
          
#calculate accuracy of datasets
def calculate_accuracy(dataframe_temp, t_variable, datasets_list):
    df_accuracy = pd.DataFrame()

    for dataset in datasets_list:
        dataset_accuracy = {'dataset': dataset}

        # Dynamically reference temperature columns
        grid_col = f'{dataset}_{t_variable}'
        insitu_col = f'{t_variable}'
        
        # Check if dataset was successfully extracted
        if grid_col not in dataframe_temp.columns:
            print(f"Skipping {dataset} - column not found.")
            continue

        temp_grid_array = dataframe_temp[grid_col].values
        temp_insitu_array = dataframe_temp[insitu_col].values
    
        # Drop null values
        valid_indices = np.logical_and.reduce([
            ~np.isnan(temp_grid_array),
            ~np.isnan(temp_insitu_array)
        ])
    
        observations = np.sum(valid_indices)
        dataset_accuracy['observations_n'] = observations

        if observations == 0:
            continue

        temp_grid_array = temp_grid_array[valid_indices]
        temp_insitu_array = temp_insitu_array[valid_indices]

        # Calculate metrics
        dataset_accuracy['correlation'] = he.spearman_r(temp_grid_array, temp_insitu_array)
        dataset_accuracy['nrmse'] = he.nrmse_mean(temp_grid_array, temp_insitu_array)
        dataset_accuracy['mean_bias'] = np.mean(temp_grid_array - temp_insitu_array)
        dataset_accuracy['rsd'] = (np.std(temp_grid_array)) / (np.std(temp_insitu_array))

        df_dataset_accuracy = pd.DataFrame([dataset_accuracy])
        df_accuracy = pd.concat([df_accuracy, df_dataset_accuracy], ignore_index=True)

    # Normalization
    df_accuracy['abs_bias'] = df_accuracy['mean_bias'].abs()
    df_accuracy['diff_rsd'] = (df_accuracy['rsd'] - 1).abs()

    df_accuracy_normalized = df_accuracy.copy().drop(columns=['mean_bias'])

    metrics = ['correlation', 'nrmse', 'abs_bias', 'diff_rsd']
    for column in metrics:
        df_accuracy_normalized[column] = normalize_column(df_accuracy_normalized[column])

    # Invert correlation (1 = ideal, 0 = worst)
    df_accuracy_normalized['correlation'] = 1 - df_accuracy_normalized['correlation']

    # Calculate CPI
    df_accuracy_normalized['cpi'] = (df_accuracy_normalized['correlation'] + 
                                     df_accuracy_normalized['nrmse'] + 
                                     df_accuracy_normalized['abs_bias'] + 
                                     df_accuracy_normalized['diff_rsd'])
    df_accuracy_normalized['cpi_rank'] = df_accuracy_normalized['cpi'].rank(ascending=True)

    df_accuracy_final = df_accuracy_normalized[['dataset','cpi','cpi_rank']].merge(
        df_accuracy[['dataset','correlation','nrmse','mean_bias','rsd', 'observations_n']], 
        on=['dataset'], 
        how='inner'
    )
    return df_accuracy_final

In [10]:
results_tmax = calculate_accuracy(df_validation_tmax, "tmax", datasets)
results_tmax

,dataset,cpi,cpi_rank,correlation,nrmse,mean_bias,rsd,observations_n
0,ERA5L,2.191287,4.0,0.777893,0.204867,-3.694137,0.946380,40132
1,HAR,2.039879,3.0,0.758532,0.199729,-3.268523,0.997151,40132
2,CHIRTS,0.109149,1.0,0.896202,0.100516,-0.014242,0.944136,40132
3,IS_K,3.069606,5.0,0.690246,0.304199,-0.270386,1.488561,40132
4,DS,0.536658,2.0,0.885611,0.106888,0.710475,0.868557,40132


In [11]:
results_tmin = calculate_accuracy(df_validation_tmin, "tmin", datasets)
results_tmin

,dataset,cpi,cpi_rank,correlation,nrmse,mean_bias,rsd,observations_n
0,ERA5L,1.409690,4.0,0.841421,0.275795,-1.111110,0.950495,40073
1,HAR,0.457993,2.0,0.862879,0.236280,0.118262,0.935333,40073
2,CHIRTS,1.064201,3.0,0.913796,0.224641,1.391532,1.008097,40073
3,IS_K,3.524156,5.0,0.721505,0.644978,0.785655,1.556826,40073
4,DS,0.444062,1.0,0.910338,0.195804,0.371178,0.867098,40073
